In [2]:

# ============================================
# Simple RAG with BM25 + CrossEncoder Reranking
# LLM: GPT-4o Mini
# Google Colab Ready
# ============================================

# Install dependencies
!pip -q install openai rank_bm25 sentence-transformers python-dotenv

from google.colab import userdata
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from openai import OpenAI

# --------------------------------------------
# Load API key from Colab Secrets
# Create a secret named: OPENAI_API_KEY
# --------------------------------------------
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

# --------------------------------------------
# Example Knowledge Base
# Replace with your own chunks/documents
# --------------------------------------------
documents = [
    "Paris is the capital city of France.",
    "London is the capital city of the United Kingdom.",
    "Berlin is the capital city of Germany.",
    "Tokyo is the capital city of Japan.",
    "The Eiffel Tower is located in Paris.",
    "France is famous for wine, cheese, and art."
]

# --------------------------------------------
# Build BM25 Index
# --------------------------------------------
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

# --------------------------------------------
# Load Cross Encoder for reranking
# --------------------------------------------
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# --------------------------------------------
# Retrieval Function
# --------------------------------------------
def retrieve(query, bm25_k=5, final_k=3):
    tokenized_query = query.lower().split()

    # BM25 scores
    scores = bm25.get_scores(tokenized_query)

    # Top BM25 candidates
    ranked = sorted(
        zip(documents, scores),
        key=lambda x: x[1],
        reverse=True
    )[:bm25_k]

    candidates = [doc for doc, _ in ranked]

    # CrossEncoder reranking
    pairs = [[query, doc] for doc in candidates]
    rerank_scores = reranker.predict(pairs)

    reranked = sorted(
        zip(candidates, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc for doc, _ in reranked[:final_k]]

# --------------------------------------------
# RAG Function
# --------------------------------------------
def ask_rag(question):
    context_docs = retrieve(question)
    context = "\n\n".join(context_docs)

    prompt = f"""
Use only the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Answer only from the provided context. If the answer is not present, say you don't know."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
    )

    return response.choices[0].message.content





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieved Context:
1. The Eiffel Tower is located in Paris.
2. Paris is the capital city of France.
3. London is the capital city of the United Kingdom.

Answer:
The Eiffel Tower is located in Paris.


In [5]:
# --------------------------------------------
# Example
# --------------------------------------------
question = "Where is the effel located?"

print("Retrieved Context:")
for i, doc in enumerate(retrieve(question), 1):
    print(f"{i}. {doc}")

print("\nAnswer:")
print(ask_rag(question))

Retrieved Context:
1. The Eiffel Tower is located in Paris.
2. Berlin is the capital city of Germany.
3. Paris is the capital city of France.

Answer:
The Eiffel Tower is located in Paris.


In [6]:

chunks = [
    {
        "text": "Paris is the capital city of France.",
        "source": "geography.pdf",
        "chunk_id": 1,
    },
    {
        "text": "The Eiffel Tower is located in Paris.",
        "source": "travel_notes.pdf",
        "chunk_id": 2,
    },
    {
        "text": "France is famous for wine and cheese.",
        "source": "geography.pdf",
        "chunk_id": 3,
    },
    {
        "text": "Tokyo is the capital of Japan.",
        "source": "asia.pdf",
        "chunk_id": 4,
    },
]

# -----------------------------
# BM25 Index
# -----------------------------
tokenized_docs = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_docs)

# -----------------------------
# Cross Encoder
# -----------------------------
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# -----------------------------
# Retrieve + Rerank
# -----------------------------
def retrieve(query, bm25_k=5, final_k=3):
    query_tokens = query.lower().split()

    scores = bm25.get_scores(query_tokens)

    ranked_idx = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )[:bm25_k]

    candidates = [chunks[i] for i in ranked_idx]

    pairs = [[query, c["text"]] for c in candidates]
    rerank_scores = reranker.predict(pairs)

    reranked = sorted(
        zip(candidates, rerank_scores),
        key=lambda x: x[1],
        reverse=True,
    )

    return [item for item, _ in reranked[:final_k]]

# -----------------------------
# RAG
# -----------------------------
def ask_rag(question):
    retrieved = retrieve(question)

    context = "\n\n".join(
        [
            f"[Source: {c['source']} | Chunk: {c['chunk_id']}]\n{c['text']}"
            for c in retrieved
        ]
    )

    prompt = f"""
Use only the context below.

{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer only using the supplied context. "
                    "If the answer is not present, say you don't know."
                ),
            },
            {"role": "user", "content": prompt},
        ],
    )

    return response.choices[0].message.content, retrieved

# -----------------------------
# Example
# -----------------------------
question = "Where is the Eiffel Tower located?"

answer, retrieved_chunks = ask_rag(question)

print("Retrieved Chunks:\n")

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"Rank {i}")
    print(f"Document : {chunk['source']}")
    print(f"Chunk ID : {chunk['chunk_id']}")
    print(f"Text      : {chunk['text']}")
    print("-" * 50)

print("\nFinal Answer:\n")
print(answer)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retrieved Chunks:

Rank 1
Document : travel_notes.pdf
Chunk ID : 2
Text      : The Eiffel Tower is located in Paris.
--------------------------------------------------
Rank 2
Document : geography.pdf
Chunk ID : 1
Text      : Paris is the capital city of France.
--------------------------------------------------
Rank 3
Document : asia.pdf
Chunk ID : 4
Text      : Tokyo is the capital of Japan.
--------------------------------------------------

Final Answer:

The Eiffel Tower is located in Paris.


In [7]:

# ==========================================================
# RAG Pipeline
# - Upload PDF
# - Chunk Document
# - BM25 Retrieval
# - Semantic Retrieval
# - CrossEncoder Reranking
# - GPT-4o-mini Generation
# ==========================================================

!pip -q install openai rank_bm25 sentence-transformers pymupdf

from google.colab import files, userdata
import fitz  # PyMuPDF
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

# -----------------------------
# Load OpenAI API Key
# -----------------------------
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# -----------------------------
# Upload PDF
# -----------------------------
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)

# -----------------------------
# Read PDF
# -----------------------------
doc = fitz.open(pdf_path)

text = ""
for page_num, page in enumerate(doc):
    text += page.get_text()

doc.close()

# -----------------------------
# Create Chunks
# -----------------------------
CHUNK_SIZE = 500

chunks = []
start = 0
chunk_id = 1

while start < len(text):
    chunk_text = text[start:start + CHUNK_SIZE]

    chunks.append({
        "chunk_id": chunk_id,
        "source": pdf_path,
        "text": chunk_text
    })

    chunk_id += 1
    start += CHUNK_SIZE

print(f"\nCreated {len(chunks)} chunks.")

# -----------------------------
# BM25 Index
# -----------------------------
tokenized = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized)

# -----------------------------
# Semantic Model
# -----------------------------
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedder.encode(
    [c["text"] for c in chunks],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# -----------------------------
# CrossEncoder
# -----------------------------
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

# -----------------------------
# Retrieval Function
# -----------------------------
def retrieve(query):

    # ---------- BM25 ----------
    bm25_scores = bm25.get_scores(query.lower().split())

    bm25_idx = np.argsort(bm25_scores)[::-1][:5]

    bm25_results = [chunks[i] for i in bm25_idx]

    # ---------- Semantic ----------
    q_emb = embedder.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    semantic_scores = embeddings @ q_emb

    semantic_idx = np.argsort(semantic_scores)[::-1][:5]

    semantic_results = [chunks[i] for i in semantic_idx]

    # ---------- Print BM25 ----------
    print("\n==============================")
    print("TOP BM25 CHUNKS")
    print("==============================")

    for rank, item in enumerate(bm25_results, 1):
        print(f"\nRank {rank}")
        print("Chunk ID:", item["chunk_id"])
        print("Source:", item["source"])
        print(item["text"][:300])

    # ---------- Print Semantic ----------
    print("\n==============================")
    print("TOP SEMANTIC CHUNKS")
    print("==============================")

    for rank, item in enumerate(semantic_results, 1):
        print(f"\nRank {rank}")
        print("Chunk ID:", item["chunk_id"])
        print("Source:", item["source"])
        print(item["text"][:300])

    # ---------- Merge ----------
    merged = {}

    for item in bm25_results + semantic_results:
        merged[item["chunk_id"]] = item

    merged = list(merged.values())

    # ---------- Rerank ----------
    pairs = [[query, x["text"]] for x in merged]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(merged, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [x[0] for x in ranked[:3]]

# -----------------------------
# Ask Question
# -----------------------------
question = input("\nEnter your question: ")

top_chunks = retrieve(question)

print("\n==============================")
print("FINAL RERANKED CHUNKS")
print("==============================")

for rank, item in enumerate(top_chunks, 1):
    print(f"\nRank {rank}")
    print("Chunk ID:", item["chunk_id"])
    print("Source:", item["source"])
    print(item["text"][:300])

context = "\n\n".join([c["text"] for c in top_chunks])

response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided context. If the answer is not present, say you don't know."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
)

print("\n==============================")
print("GPT-4o-mini ANSWER")
print("==============================")
print(response.choices[0].message.content)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 33.0 MB/s eta 0:00:00


Saving RAG_Complete_Expert_Reference.pdf to RAG_Complete_Expert_Reference.pdf
Uploaded: RAG_Complete_Expert_Reference.pdf

Created 117 chunks.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Enter your question: what is the document about

TOP BM25 CHUNKS

Rank 1
Chunk ID: 53
Source: RAG_Complete_Expert_Reference.pdf
 cosine similarity.
Typical top-K values: 3-10 for direct generation, 20-100 for reranking pipelines.
7.2 Query Transformation Techniques
The user's raw query is often a poor retrieval signal. Query transformation improves recall by
creating better search queries.
HyDE — Hypothetical Document Embedd

Rank 2
Chunk ID: 3
Source: RAG_Complete_Expert_Reference.pdf
mitations that RAG directly addresses:
•
Knowledge cutoff: LLMs only know what was in their training data. They cannot answer
questions about events after their cutoff date.
•
Hallucination: Without a grounding signal, LLMs confidently fabricate plausible-
sounding but incorrect facts.
•
No access t

Rank 3
Chunk ID: 31
Source: RAG_Complete_Expert_Reference.pdf
e.
Analogy: The Library Map
Imagine a library where every book has a GPS coordinate based on its content. Books about dogs
and books about pets 